# TIA robustness run (self-contained)

Four checks for the paper, all on the frozen features and the frozen training code of TIA:
fidelity of a re-run (R1), out-of-fold probabilities on the training targets for the COUNT diagnostic (R2), a
source-signature ablation (R3) and a training run without the binning transformations (R4).

**Everything is inside this notebook.** You don't need a Google Drive folder or any uploads.

**How to run**
1. Open this notebook in Colab (File › Upload notebook). A CPU runtime is enough.
2. Runtime › **Run all**. It takes about 10 min; step 3 prints one line per information condition.
3. `robustness_results.zip` downloads at the end. If the browser blocks the download, open the Files panel (folder
   icon on the left), right-click `robustness_results.zip` › Download. Put the zip in the project folder.

If the runtime disconnects or restarts, use Runtime › Run all again. Every step is safe to repeat.

## 1. Unpack the bundle

The first cell holds the embedded code and data (hidden); the second unpacks them and checks every file against its checksum.

In [ ]:
#@title Embedded bundle: code and data, ~127 kB (hidden) { display-mode: "form" }
PAYLOAD_SHA256 = "5b2c8605d28f8fcd09d6b5ccc40cf6b52ca9f546f4d84b0f365752f3f475ff91"
FILE_SHA256 = {
 "config.json": "baf0d926c0fd26707bb59e412202d9c15b145382e3203fae4854b1f2f6663cfe",
 "inputs/features.csv": "a6c9072d3dbead2134ecbd47c7b6e105937a1896736f77c9e6794c78164c2755",
 "inputs/frozen_test.csv": "b578f3db9359d0fc593c5105b19caa9194c385fdda90571540bcf8acdc91f212",
 "inputs/manifest.csv": "a1dcde336cc003f596d848537b96f5ebd531eb3d9e989f68d745e2dcc78e9fa5",
 "lexicons.py": "7e2dc5fe1e2aa1cff58a2f7cf6075c17317c929e1c956f50d1bd2ae296ebc7ad",
 "method_lib.py": "4f89d20ff097f6c545ef5c15bc007a21712bb7f9a80c06123213787fa5234d53",
 "robustness_lib.py": "d9f8ed4288c7fce9a6f3332d275d7c510535f1944d48327bdb1d332616a58960"
}
PAYLOAD = """
...
"""

In [ ]:
import base64, hashlib, io, tarfile
from pathlib import Path

WORK = Path("/content/robustness") if Path("/content").is_dir() else Path.cwd() / "robustness_run"
raw = base64.b64decode("".join(PAYLOAD.split()))
assert hashlib.sha256(raw).hexdigest() == PAYLOAD_SHA256, "the embedded bundle is damaged: download the notebook again"
WORK.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(raw), mode="r:xz") as tar:
    if hasattr(tarfile, "data_filter"):
        tar.extractall(WORK, filter="data")
    else:
        tar.extractall(WORK)
for name, sha in FILE_SHA256.items():
    assert hashlib.sha256((WORK / name).read_bytes()).hexdigest() == sha, f"{name} does not match its checksum"
print(f"unpacked to {WORK}: {len(FILE_SHA256)} files, checksums ok")

## 2. Check the runtime

In [ ]:
import importlib, json, sys
assert "WORK" in globals(), "run the cells from the top: Runtime › Run all"
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
import method_lib, robustness_lib as rb
importlib.reload(method_lib); importlib.reload(rb)
CFG = json.loads((WORK / "config.json").read_text())
print("ok", rb.versions())

## 3. Run R1–R4 (about 10 min)

The report at the end shows the fidelity check: the largest difference between the re-run and the frozen test probabilities, and the share of identical top-1 answers.

In [ ]:
assert "rb" in globals(), "run the cells from the top: Runtime › Run all"
report = rb.run(WORK, WORK / "results", CFG)

## 4. Download the results

In [ ]:
import shutil
assert (WORK / "results" / "report.json").exists(), "step 3 has not finished: run all cells from the top"
shutil.copy(WORK / "config.json", WORK / "results" / "config.json")
zip_path = shutil.make_archive(str(WORK.parent / "robustness_results"), "zip", WORK / "results")
print("wrote", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass  # not on Colab: the zip is at the path above
except Exception as e:  # noqa: BLE001
    print(f"automatic download failed ({e}). Files panel › right-click robustness_results.zip › Download")